# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hanizakkk/flyrank_working-repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Logistic Regression as an interpretable reference (standardized
features, `max_iter=1000`, `random_state=42`), and Random Forest
(`n_estimators=300, max_depth=6, random_state=42`) as the final model. Both
are readable enough to explain to a non-technical stakeholder ("which
features push the score up"), and Random Forest handles the non-linear
interaction between `avg_position`, `ctr`, and `impressions` that the linear
model can't capture on its own — which is exactly why it edges out logistic
regression below.

In [ ]:
# Run in Colab: capstone_full_pipeline.ipynb, section 4.
import json
with open("../outputs/model_metrics.json") as f:
    model_metrics = json.load(f)
print(json.dumps(model_metrics, indent=2))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split:** grouped by `client_hash_id` — 80/20 client holdout, `seed=42`. A
client's pages never appear in both train and test. This matters here because
a random row split let the model see some of a client's other pages during
training and partly infer that client's typical traffic level rather than
learn a transferable signal - confirmed directly in ML-09's split comparison
(random split: precision@50 = 0.96; client holdout: precision@50 = 0.86 -
the ~10-point gap is the cost of the shortcut a random split allows).

In [ ]:
with open("../outputs/split_comparison.json") as f:
    split_comparison = json.load(f)
print(json.dumps(split_comparison, indent=2))
# client_overlap=0 confirms the grouped split holds - no client appears in
# both train and test.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same split, same metric as the ML-07 baseline.** Result:

| Model | Precision@50 |
|---|---|
| Baseline rule | 0.70 |
| Logistic Regression | 0.84 |
| Random Forest | **0.86** |

(test-split base rate: 66.2% future decline). Both models beat the baseline
rule on the identical held-out clients; Random Forest is the final model
carried into ML-09/ML-10.

In [ ]:
# Same numbers as above, straight from the real run's model_metrics.json:
import pandas as pd
pd.DataFrame(model_metrics)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Feature importance** (Random Forest, see `work/figures/feature_importance.svg`):
`avg_position` and `ctr` dominate - consistent with the baseline rule's own
logic, which is reassuring rather than surprising: the model rediscovered
roughly the same signal a human wrote down by hand, then combined it more
precisely than a fixed threshold could.

**Where it's wrong:** the confident-but-wrong cases cluster around items with
moderate impressions and a position hovering right at the page-one/page-two
boundary (avg_position ~9-11) - exactly where the `avg_position >= 10` rule
threshold (and the model's own decision boundary) is least stable. This is a
real limitation: items near that boundary should be treated as
lower-confidence flags, not auto-actioned.

In [ ]:
# See work/figures/feature_importance.svg for the exported chart.
# Confident-but-wrong rows were inspected directly in the pipeline notebook
# (capstone_full_pipeline.ipynb, section 4, `confident_wrong` dataframe) -
# not re-exported here since they are per-item test-split rows, not an
# aggregated result safe to commit on their own.
print("See work/figures/feature_importance.svg")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.